In [1]:
###################################################################################
# Title: Var_preprocessing_REA.py

# Purpose: Preprocess importance sampling data: Regrid, Spatial and temporal selection, change calendar, and save to netCDF files.

# Author: Onno Nennecke on 03.06.2025 Modified: 08.04.2026

# Input data: 

#     - IS data lies here: /climca/data/REA_output/RL_winter_DEU/
#     - CMIP6 runs are defined in the csv file: /home/onennecke/CMIP_models/CMIP6_runs.csv
#     - Alpha mask for rescaling wind speed lies here: /home/onennecke/Capacity_data/alpha_land_sea.nc

# Output data:

#     - This file lies here: /climca/people/onennecke/not_debiased_data/
###################################################################################


# Importing libraries
import xarray as xr
import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
import os
import glob
import cftime
import time
import re
import multiprocessing



# Importing functions
import Functions.grid_func as grid_func
import Functions.wind_model_func as wind_model_func

/home/onennecke/.conda/envs/env_ma_on/lib/python3.12/site-packages/esmpy/interface/loadESMF.py:94: VersionWarning: ESMF installation version 8.8.0, ESMPy version 8.8.0b0
  warnings.warn("ESMF installation version {}, ESMPy version {}".format(


In [2]:
# Import alpha mask for rescaling wind speed
alpha_mask = xr.open_dataset('/home/onennecke/Capacity_data/alpha_land_sea.nc')

# Read the dataframe from the csv file
df = pd.read_csv("/home/onennecke/CMIP_models/CESM2_LE_runs.csv", dtype=str)

# Load climate data
# variables = ['U10', 'FSDS', 'TREFHT', 'TREFHTMX'] # List of variables
variables = ['sfcWind', 'rsds', 'tas', 'tasmax'] # List of variables

# variables = ['U10', 'FSDS', 'TREFHT', 'TREFHTMX', 'PSL'] # List of variables
# var_dict = {'sfcWind': 'U10', 'rsds': 'FSDS', 'tas': 'TREFHT', 'tasmax': 'TREFHTMX', 'psl': 'PSL'}
# var_dict = {'U10': 'sfcWind', 'FSDS': 'rsds', 'TREFHT': 'tas', 'TREFHTMX': 'tasmax', 'PSL': 'psl'}
# var_dict = {'sfcWind-DEU': 'sfcWind', 'rsds-DEU': 'rsds', 'tas-DEU': 'tas', 'tasmax-DEU': 'tasmax', 'psl-atleu': 'psl'}
var_dict = {'sfcWind': 'sfcWind-DEU', 'rsds': 'rsds-DEU', 'tas': 'tas-DEU', 'tasmax': 'tasmax-DEU', 'psl': 'psl-atleu'}


In [ ]:
'''# nc = xr.open_dataset("/climca/data/CESM2_LE/U10/day_raw/b.e21.BSSP370smbb.f09_g17.LE2-1011.001.cam.h1.U10.20450101-20541231.nc")
nc = xr.open_dataset("/climca/data/REA_output/RL_winter_DEU/NCAR/CESM2/ssp370-2025-x2/day/atmos/rsds-DEU/ens001/rsds-DEU_day_CESM2_ssp370-2025-x2_ens001_2025.nc")
nc = nc[['rsds']]
nc.load()'''

<xarray.Dataset> Size: 285kB
Dimensions:  (time: 154, lat: 27, lon: 17)
Coordinates:
  * lat      (lat) float64 216B 40.05 40.99 41.94 42.88 ... 62.67 63.61 64.55
  * lon      (lon) float64 136B 0.0 1.25 2.5 3.75 5.0 ... 16.25 17.5 18.75 20.0
    sim      <U90 360B 'c1.000.000.001.001.001.002.003.000.000.000.000.000.00...
  * time     (time) datetime64[ns] 1kB 2025-10-02 2025-10-03 ... 2026-03-04
Data variables:
    rsds     (time, lat, lon) float32 283kB 151.8 153.5 153.0 ... 64.15 88.89
Attributes: (12/15)
    Conventions:             CF-1.0
    source:                  CAM
    case:                    c1_000
    logname:                 u290372
    host:                    
    initial_file:            /work/bb1152/u290372/cesm215_input_data/atm/cam/...
    ...                      ...
    simulation_name:         c1.000.000.001.001.001.002.003.000.000.000.000.0...
    initial_condition:       /work/bb1152/u290372/cesm215_archive/GKLT/prep_s...
    initial_condition_year:  0400_2024
    compset:                 BSSP370cmip6
    readme:                  https://github.com/peterpeterp/REA_low_energy_wi...
    preprocessing:           gib rectangle araound Germany (lon: 0 to 20 lat:...

In [ ]:
'''files = ["/climca/data/CESM2_LE/U10/day_raw/b.e21.BSSP370smbb.f09_g17.LE2-1011.001.cam.h1.U10.20350101-20441231.nc", "/climca/data/CESM2_LE/U10/day_raw/b.e21.BSSP370smbb.f09_g17.LE2-1011.001.cam.h1.U10.20450101-20541231.nc"]
nc = xr.open_mfdataset(files, preprocess=grid_func.preprocess)
nc = nc[['U10']]
nc.load()
'''

<xarray.Dataset> Size: 5MB
Dimensions:  (time: 7300, lat: 16, lon: 10)
Coordinates:
  * lat      (lat) float64 128B 45.71 46.65 47.59 48.53 ... 57.96 58.9 59.84
  * lon      (lon) float64 80B 5.0 6.25 7.5 8.75 10.0 ... 12.5 13.75 15.0 16.25
  * time     (time) object 58kB 2035-01-01 00:00:00 ... 2054-12-31 00:00:00
Data variables:
    U10      (time, lat, lon) float32 5MB 2.091 1.426 1.381 ... 3.33 3.313 3.643
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    case:              b.e21.BSSP370smbb.f09_g17.LE2-1011.001
    logname:           sunseon
    host:              mom2
    initial_file:      b.e21.BHISTsmbb.f09_g17.LE2-1011.001.cam.i.2015-01-01-...
    topography_file:   /mnt/lustre/share/CESM/cesm_input/atm/cam/topo/fv_0.9x...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1

In [5]:
df
# Show all datatypes of the columns in the dataframe
# df.dtypes

,ESM,Institution,run
0,CESM2,NCAR,LE2-1001_001
1,CESM2,NCAR,LE2-1011_001
2,CESM2,NCAR,LE2-1021_002
3,CESM2,NCAR,LE2-1031_002
4,CESM2,NCAR,LE2-1041_003
...,...,...,...
95,CESM2,NCAR,LE2-1301_016
96,CESM2,NCAR,LE2-1301_017
97,CESM2,NCAR,LE2-1301_018
98,CESM2,NCAR,LE2-1301_019


In [3]:
df
# Show all datatypes of the columns in the dataframe
# df.dtypes

,ESM,Institution,run
0,CESM2,NCAR,LE2-1001_001
1,CESM2,NCAR,LE2-1011_001
2,CESM2,NCAR,LE2-1021_002
3,CESM2,NCAR,LE2-1031_002
4,CESM2,NCAR,LE2-1041_003
...,...,...,...
95,CESM2,NCAR,LE2-1301_016
96,CESM2,NCAR,LE2-1301_017
97,CESM2,NCAR,LE2-1301_018
98,CESM2,NCAR,LE2-1301_019


In [4]:
ESM = 'CESM2'
Inst = 'NCAR'
ESM_name = 'CESM2_REA'
scenario = 'ssp370-2025-x2'
time_res = 'day'
mod = 'atmos'
variables_folders = ['sfcWind-DEU', 'rsds-DEU', 'tas-DEU', 'tasmax-DEU'] # List of variables
variables = ['sfcWind', 'rsds', 'tas', 'tasmax'] # List of variables
ens_mems = [f"ens{i:03}" for i in range(127)]


# for i in range(len(df[0:2])):
def one_run(i):

    run_time = time.time()
    ensemble_member = ens_mems[i]
    
    ESM_run = f'{ESM}_{ensemble_member}'
    

    print(f'Processing Run Nr. {i}, {ensemble_member}\n')
    
    for var in variables:
        # output_var = var_dict[var]
        output_file = f'/climca/people/onennecke/not_debiased_data/REA/CESM2_LE_REA_{ensemble_member}_{var}.nc'

        if os.path.isfile( output_file ) == False:
            
            print(f'Processing variable: {var}')
            file = f'/climca/data/REA_output/RL_winter_DEU/NCAR/CESM2/ssp370-2025-x2/day/atmos/{var}-DEU/{ensemble_member}/{var}-DEU_day_CESM2_ssp370-2025-x2_{ensemble_member}_2025.nc'
            # print(path)
            # path = f'/climca/data/CESM2_LE/{var}/day_raw/b.e21.{experiment}.f09_g17.{LE_ID_frcng}.{ensemble_member}.cam.h1.{var}.20150101-20241231.nc' 
            # files = [f for f in glob.glob(path) if f.endswith('.nc')]

            nc = xr.open_mfdataset(file, preprocess=grid_func.preprocess)
            nc = nc[[var]]
            nc = grid_func.regrid(nc, s = 47, n = 56, w = 6, e = 16)  # Regrid the data
            
            nc = nc.drop_vars('height') if 'height' in nc.coords else nc
            
            # nc = nc.rename({var: var_dict[var]})
            
            if var == 'sfcWind':
                nc = wind_model_func._wind_scale(nc, 100, alpha_mask['mask'], 10)
            
            nc = nc.assign_coords(ESM=ESM)  # Assign ESM coordinate
            nc = nc.assign_coords(run=ensemble_member)  # Assign run coordinate
            nc = nc.assign_coords(ESM_run=ESM_run)  # Assign ESM_run coordinate
            
            nc = nc.load()
            
            # Save the dataset
            nc.to_netcdf(output_file)

            print('Run time: ', int(np.floor((time.time()  - run_time) / 60)),'m', round((time.time()  - run_time) % 60,1),'s', '\n')
        else:
            print(f'File {output_file} already exists. Skipping variable {var} for run {ensemble_member}.\n')

    
    var = 'psl'
    # output_var = 'psl'
    
    output_file = f'/climca/people/onennecke/not_debiased_data/REA/CESM2_LE_REA_{ensemble_member}_{var}.nc'

    if os.path.isfile( output_file ) == False:

        run_time = time.time()

        print(f'Processing variable: {var}')
        file = f'/climca/data/REA_output/RL_winter_DEU/NCAR/CESM2/ssp370-2025-x2/day/atmos/{var}-atleu/{ensemble_member}/{var}-atleu_day_CESM2_ssp370-2025-x2_{ensemble_member}_2025.nc'
        nc = xr.open_mfdataset(file, preprocess=grid_func.preprocess)
        # files = [f for f in glob.glob(path) if f.endswith('.nc')]


        # nc = xr.open_mfdataset(files, preprocess=grid_func.preprocess_psl)
        nc = nc[[var]]
        nc = grid_func.regrid(nc, s = 30, n = 70, w = 340, e = 30)

        nc = nc.drop_vars('height') if 'height' in nc.coords else nc
        # nc = nc.rename({var: var_dict[var]})

        nc = nc.assign_coords(ESM=ESM)  # Assign ESM coordinate
        nc = nc.assign_coords(run=ensemble_member)  # Assign run coordinate
        nc = nc.assign_coords(ESM_run=ESM_run)  # Assign ESM_run coordinate
        
        nc = nc.load()

        # Save the dataset
        nc.to_netcdf(output_file)

        print('Run time: ', int(np.floor((time.time()  - run_time) / 60)),'m', round((time.time()  - run_time) % 60,1),'s')
    else:
        print(f'File {output_file} already exists. Skipping variable {var} for run {ensemble_member}.\n')





# p = multiprocessing.Pool(64)
# p.map(one_run, range(len(df)))


In [5]:
one_run(1)

Processing Run Nr. 1, ens001

Processing variable: sfcWind
Run time:  0 m 5.5 s 

Processing variable: rsds
Run time:  0 m 5.6 s 

Processing variable: tas
Run time:  0 m 5.8 s 

Processing variable: tasmax
Run time:  0 m 5.9 s 

Processing variable: psl
Run time:  0 m 0.3 s


In [6]:
for i in range(1,127):
    one_run(i)
    # print(i)

Processing Run Nr. 1, ens001

File /climca/people/onennecke/not_debiased_data/REA/CESM2_LE_REA_ens001_sfcWind.nc already exists. Skipping variable sfcWind for run ens001.

File /climca/people/onennecke/not_debiased_data/REA/CESM2_LE_REA_ens001_rsds.nc already exists. Skipping variable rsds for run ens001.

File /climca/people/onennecke/not_debiased_data/REA/CESM2_LE_REA_ens001_tas.nc already exists. Skipping variable tas for run ens001.

File /climca/people/onennecke/not_debiased_data/REA/CESM2_LE_REA_ens001_tasmax.nc already exists. Skipping variable tasmax for run ens001.

File /climca/people/onennecke/not_debiased_data/REA/CESM2_LE_REA_ens001_psl.nc already exists. Skipping variable psl for run ens001.

Processing Run Nr. 2, ens002

Processing variable: sfcWind
Run time:  0 m 0.1 s 

Processing variable: rsds
Run time:  0 m 0.3 s 

Processing variable: tas
Run time:  0 m 0.4 s 

Processing variable: tasmax
Run time:  0 m 0.5 s 

Processing variable: psl
Run time:  0 m 0.1 s
Processi